In [1]:
!pip3 install matplotlib
!pip3 install pandas

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [2]:
folder_of_data = "/data001/projects/enserrog/AbigailData/energy_over_time/energy_decomposition"

In [3]:
import matplotlib.pyplot as plt
import pandas as pd
import os

# load files

In [4]:
def find_file_recursive(root_folder, file_name, current_depth=0, max_depth=None, verbose=True):
    """
    Recursively search for a file in all subdirectories
    
    Args:
        root_folder (str): Path to start searching from
        file_name (str): Name of the file to look for
        current_depth (int): Current recursion depth (for display purposes)
        max_depth (int): Maximum depth to search (None for unlimited)
        verbose (bool): Whether to print detailed output
    
    Returns:
        list: List of paths where the file was found
    """
    found_paths = []
    indent = "  " * current_depth
    
    # Check if we've reached max depth
    if max_depth is not None and current_depth > max_depth:
        return found_paths
    
    # Normalize the path to handle different path separators
    root_folder = os.path.normpath(root_folder)
    
    # Check if the folder exists and is accessible
    if not os.path.exists(root_folder):
        if verbose:
            print(f"{indent}❌ Folder does not exist: {root_folder}")
        return found_paths
    
    if not os.path.isdir(root_folder):
        if verbose:
            print(f"{indent}❌ Not a directory: {root_folder}")
        return found_paths
    
    try:
        items = os.listdir(root_folder)
    except PermissionError:
        if verbose:
            print(f"{indent}❌ Permission denied: {root_folder}")
        return found_paths
    except Exception as e:
        if verbose:
            print(f"{indent}❌ Error accessing {root_folder}: {e}")
        return found_paths
    
    if verbose:
        print(f"{indent}📁 Searching in: {root_folder}")
    
    # Check if the file exists in current directory
    file_path = os.path.join(root_folder, file_name)
    if os.path.isfile(file_path):  # Use isfile instead of exists to ensure it's a file
        if verbose:
            print(f"{indent}  ✅ Found '{file_name}'")
        found_paths.append(file_path)
    else:
        if verbose:
            print(f"{indent}  ❌ '{file_name}' not found")
    
    # Recursively search in subdirectories
    subdirs = []
    for item in items:
        item_path = os.path.join(root_folder, item)
        if os.path.isdir(item_path):
            subdirs.append(item_path)
    
    if verbose and subdirs:
        print(f"{indent}  📂 Found {len(subdirs)} subdirectories")
    
    for subdir in subdirs:
        try:
            subfolder_results = find_file_recursive(subdir, file_name, current_depth + 1, max_depth, verbose)
            found_paths.extend(subfolder_results)
        except Exception as e:
            if verbose:
                print(f"{indent}  ❌ Error processing {subdir}: {e}")
    
    return found_paths

In [5]:
def extract_results_folder(file_path):
    """
    Extract the folder name that ends with '_results' from a file path
    
    Args:
        file_path (str): Full path to a file
        
    Returns:
        str: The folder name ending with '_results', or None if not found
    """
    # Split the path into components
    path_parts = file_path.split(os.sep)
    
    # Look for a folder that ends with '_results'
    for part in path_parts:
        if part.endswith('_results'):
            return part
    
    return None

# find all reaches and begin generating images

In [6]:
all_reach_states = find_file_recursive(folder_of_data, "per_reach_state.csv")

📁 Searching in: /data001/projects/enserrog/AbigailData/energy_over_time/energy_decomposition
  ❌ 'per_reach_state.csv' not found
  📂 Found 24 subdirectories
  📁 Searching in: /data001/projects/enserrog/AbigailData/energy_over_time/energy_decomposition/210421_results
    ❌ 'per_reach_state.csv' not found
    📂 Found 1 subdirectories
    📁 Searching in: /data001/projects/enserrog/AbigailData/energy_over_time/energy_decomposition/210421_results/210421_rep1
      ❌ 'per_reach_state.csv' not found
      📂 Found 4 subdirectories
      📁 Searching in: /data001/projects/enserrog/AbigailData/energy_over_time/energy_decomposition/210421_results/210421_rep1/begin_reach
        ❌ 'per_reach_state.csv' not found
      📁 Searching in: /data001/projects/enserrog/AbigailData/energy_over_time/energy_decomposition/210421_results/210421_rep1/mid_reach
        ❌ 'per_reach_state.csv' not found
      📁 Searching in: /data001/projects/enserrog/AbigailData/energy_over_time/energy_decomposition/210421_results

In [56]:
import numpy as np
import matplotlib.pyplot as plt
def run_experiment(df, output_dir,reach_idx="" , mouse_id="", title_prefix="", show_mid_point=True):
    
    unique_stims = df['stim'].unique()
    unique_reaches = df['reach_idx'].unique()
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Results storage
    results = []
    
    # Plot for each stimulus
    for stim in unique_stims:
        stim_data = df[df['stim'] == stim]
        unique_reaches_stim = stim_data['reach_idx'].unique()
        
        # Filter by specific reach_idx if provided
        if reach_idx:
            unique_reaches_stim = [r for r in unique_reaches_stim if r == reach_idx]
        
        for reach in unique_reaches_stim:
            reach_data = stim_data[stim_data['reach_idx'] == reach]
            
            if reach_data.empty:
                continue
                
            # Assume we have time column for defining ranges
            x1 = 350
            x2 = 450

            # parse out range
            reach_data_sliced = reach_data.iloc[x1:x2] 
            
            # Get velocity over x (assuming this means derivative of x position)
            if 'x' in reach_data_sliced.columns:
                x_data = reach_data_sliced['x'].values
                firingRate_rest = reach_data.iloc[:x1]['firing_rate'].values
                firingRate = reach_data_sliced['firing_rate'].values
                if len(x_data) > 1:
                    
                    # velocity
                    velocity_x = np.gradient(x_data)
                    reach_data_sliced = reach_data_sliced.copy()
                    reach_data_sliced['velocity_x'] = velocity_x

                    # acceleration
                    acceleration_x = np.gradient(velocity_x)
                    reach_data_sliced = reach_data_sliced.copy()
                    reach_data_sliced['acceleration_x'] = acceleration_x
                    
                else:
                    velocity_x = np.array([0])
            else:
                velocity_x = np.array([0])
            
            # Find max&min velocity in range [x1, x2]
            if len(velocity_x) > 0:
                max_velocity = np.max(np.abs(velocity_x))
                max_velocity_idx = np.argmax(np.abs(velocity_x))
                min_velocity = np.min(np.abs(velocity_x))
                min_velocity_idx = np.argmin(np.abs(velocity_x))

                # find difference relative to rest
                firing_rate_gradient_at_velocity_max = np.mean(firingRate_rest) - firingRate[max_velocity_idx]
                firing_rate_gradient_at_velocity_min = np.mean(firingRate_rest) - firingRate[min_velocity_idx]
                
            else:
                max_velocity = 0
                max_velocity_idx = 0
                min_velocity = 0
                min_velocity_idx = 0

            # Find max&min acceleration in range [x1, x2]
            if len(acceleration_x) > 0:
                max_acceleration = np.max(np.abs(acceleration_x))
                max_acceleration_idx = np.argmax(np.abs(acceleration_x))
                min_acceleration = np.min(np.abs(acceleration_x))
                min_acceleration_idx = np.argmin(np.abs(acceleration_x))
            else:
                max_acceleration = 0
                max_acceleration_idx = 0
                min_acceleration = 0
                min_acceleration_idx = 0
                
            
            # Find max energy in range [x1, x2]
            if 'energy' in reach_data_sliced.columns:
                energy_data = reach_data_sliced['energy'].values
                max_energy = np.max(energy_data)
                max_energy_idx = np.argmax(energy_data)
                min_energy = np.min(energy_data)
                min_energy_idx = np.argmin(energy_data)
            else:
                max_energy = 0
                max_energy_idx = 0
            
            # Find max/min firing rate in range [x1, x2]
            if 'firing_rate' in reach_data_sliced.columns:
                firing_rate_data = reach_data_sliced['firing_rate'].values
                max_firing_rate = np.max(firing_rate_data)
                min_firing_rate = np.min(firing_rate_data)
                max_firing_idx = np.argmax(firing_rate_data)
                min_firing_idx = np.argmin(firing_rate_data)
            else:
                max_firing_rate = 0
                min_firing_rate = 0
                max_firing_idx = 0
                min_firing_idx = 0
            
            # Store results
            result = {
                'mouse_id': mouse_id,
                'stim': stim,
                'reach_idx': reach,
                'max_velocity': max_velocity,
                'min_velocity': min_velocity,
                'max_velocity_idx': max_velocity_idx,
                'min_velocity_idx': min_velocity_idx,
                'max_acceleration': max_acceleration,
                'min_acceleration': min_acceleration,
                'max_acceleration_idx': max_acceleration_idx,
                'min_acceleration_idx': min_acceleration_idx,
                'max_energy': max_energy,
                'min_energy': min_energy,
                'max_energy_idx': max_energy_idx,
                'min_energy_idx': min_energy_idx,
                'max_firing_rate': max_firing_rate,
                'min_firing_rate': min_firing_rate,
                'max_firing_idx': max_firing_idx,
                'min_firing_idx': min_firing_idx,
                'firing_rate_gradient_velocity_max': firing_rate_gradient_at_velocity_max,
                'firing_rate_gradient_velocity_min': firing_rate_gradient_at_velocity_min
            }
            results.append(result)
    return results
        
    

In [57]:
def ensure_directory_exists(directory_path):
    """
    Check if a directory exists and create it (including all parent directories) if it doesn't.
    
    Args:
        directory_path (str): Path to the directory to check/create
    
    Returns:
        bool: True if directory exists or was created successfully, False if creation failed
    """
    try:
        os.makedirs(directory_path, exist_ok=True)
        return True
    except Exception as e:
        print(f"Error creating directory '{directory_path}': {e}")
        return False

In [58]:
all_reach_states

['/data001/projects/enserrog/AbigailData/energy_over_time/energy_decomposition/210423_results/210423_rep1/begin_reach/per_reach_state.csv',
 '/data001/projects/enserrog/AbigailData/energy_over_time/energy_decomposition/210423_results/210423_rep1/mid_reach/per_reach_state.csv',
 '/data001/projects/enserrog/AbigailData/energy_over_time/energy_decomposition/210423_results/210423_rep1/post_reach/per_reach_state.csv',
 '/data001/projects/enserrog/AbigailData/energy_over_time/energy_decomposition/210423_results/210423_rep1/full_reach/per_reach_state.csv',
 '/data001/projects/enserrog/AbigailData/energy_over_time/energy_decomposition/210425_results/210425_rep1/begin_reach/per_reach_state.csv',
 '/data001/projects/enserrog/AbigailData/energy_over_time/energy_decomposition/210425_results/210425_rep1/mid_reach/per_reach_state.csv',
 '/data001/projects/enserrog/AbigailData/energy_over_time/energy_decomposition/210425_results/210425_rep1/post_reach/per_reach_state.csv',
 '/data001/projects/enserro

In [7]:
pd.read_csv(all_reach_states[0])

,reach_idx,stim,x,y,z,firing_rate,energy,j,h
0,0,0,0.026840,0.118307,0.811070,0.133333,-0.030509,0.000000,0.030509
1,0,0,0.026830,0.118189,0.811066,0.142857,-0.010001,0.000000,0.010001
2,0,0,0.026794,0.118156,0.811135,0.125000,0.218246,0.000000,-0.218246
3,0,0,0.026739,0.118220,0.811268,0.133333,-0.000000,0.000000,0.000000
4,0,0,0.026693,0.118371,0.811452,0.120000,0.218246,0.000000,-0.218246
...,...,...,...,...,...,...,...,...,...
15495,19,2,0.044681,0.040594,0.858180,0.420000,0.217441,0.227199,-0.444641
15496,19,2,0.044708,0.040685,0.858544,0.466667,0.675992,0.000000,-0.675992
15497,19,2,0.044718,0.040825,0.858786,0.425000,0.217441,0.227199,-0.444641
15498,19,2,0.044711,0.040986,0.858941,0.400000,0.414794,0.060355,-0.475149


In [64]:
output_dir = "/data001/projects/enserrog/AbigailData/energy_over_time/energy_decomposition/summary"


results = []
for reach in all_reach_states:
    
    if not "full_reach" in reach:
        continue
    
    mouse_id = extract_results_folder(reach).split("_")[0]
    print(mouse_id)
    
    reach_data = pd.read_csv(reach)
    individual_reaches = reach_data.groupby(["reach_idx", "stim"])

    all_reach_output_dir = output_dir + f"/{mouse_id}"
    ensure_directory_exists(all_reach_output_dir)
    
    for name, group_df in individual_reaches:
        # print(name)
        results += run_experiment(group_df.reset_index(), all_reach_output_dir,reach_idx=name[0],mouse_id = mouse_id, title_prefix=f"{mouse_id}, index:{str(name[0])}")
            


210423
210425
210511
210512
210514
210515
210606
210614
220515
220516
220517
220518
220520


In [62]:
results[2]

{'mouse_id': '210423',
 'stim': np.int64(2),
 'reach_idx': np.int64(0),
 'max_velocity': np.float64(0.11078844570807889),
 'min_velocity': np.float64(1.0824140132349724e-05),
 'max_velocity_idx': np.int64(47),
 'min_velocity_idx': np.int64(4),
 'max_acceleration': np.float64(0.027668736339533068),
 'min_acceleration': np.float64(2.5491857944849594e-07),
 'max_acceleration_idx': np.int64(53),
 'min_acceleration_idx': np.int64(10),
 'max_energy': np.float64(0.6979840203766089),
 'min_energy': np.float64(-0.1588862296595383),
 'max_energy_idx': np.int64(74),
 'min_energy_idx': np.int64(86),
 'max_firing_rate': np.float64(0.6727272727272727),
 'min_firing_rate': np.float64(0.3636363636363636),
 'max_firing_idx': np.int64(90),
 'min_firing_idx': np.int64(74),
 'firing_rate_gradient_velocity_max': np.float64(-0.05572712842712846),
 'firing_rate_gradient_velocity_min': np.float64(0.03518196248196248)}

In [45]:
# pd.DataFrame(results).to_csv("/data001/projects/ising/outputs/min_max_results.csv")

In [65]:
base_reach_stats = pd.read_csv("~/Energy_over_time_hypothesis.csv")
base_reach_stats['mouse_id_normalized'] = base_reach_stats['mouse_id'].apply(lambda x: x.split("_")[0])

In [66]:
results_bridged = []

for reachs in results:
    
    out = {
        **reachs,
        **(base_reach_stats[np.logical_and.reduce([
        base_reach_stats['mouse_id_normalized'] == reachs['mouse_id'],
        base_reach_stats['stim'] == reachs['stim'],
        base_reach_stats['reach_id'] == reachs['reach_idx']
    ])].iloc[0]).to_dict()}

    del out['reach_idx']
    
    results_bridged += [
        out
    ]

In [67]:
results_bridged[0]

{'mouse_id': '210423_fChR2',
 'stim': 0,
 'max_velocity': np.float64(0.08477721562150149),
 'min_velocity': np.float64(3.163232682949768e-06),
 'max_velocity_idx': np.int64(47),
 'min_velocity_idx': np.int64(21),
 'max_acceleration': np.float64(0.021997129390192605),
 'min_acceleration': np.float64(1.1501995198309745e-05),
 'max_acceleration_idx': np.int64(56),
 'min_acceleration_idx': np.int64(83),
 'max_energy': np.float64(0.2991605858071212),
 'min_energy': np.float64(-0.2078353719553799),
 'max_energy_idx': np.int64(29),
 'min_energy_idx': np.int64(62),
 'max_firing_rate': np.float64(0.4),
 'min_firing_rate': np.float64(0.109090909090909),
 'max_firing_idx': np.int64(84),
 'min_firing_idx': np.int64(36),
 'firing_rate_gradient_velocity_max': np.float64(0.010597340754483653),
 'firing_rate_gradient_velocity_min': np.float64(-0.15303902288188004),
 'file_name': '210423_fChR2_bin10_spike.mat',
 'reach_id': 0,
 'num_neurons': 5,
 'exlcuded_reach': 0,
 'fChR2': 1,
 'fChR5': 0,
 'fChR4':

In [68]:
pd.DataFrame(results_bridged).to_csv("min_max_results_additions_fireGradient.csv")

In [108]:
!pwd

/home/ceas/enserrog/ising_EnergyOverTime
